In [1]:
import os
import pandas as pd

# Kaggle typically stores user-uploaded files in /kaggle/input/ or /kaggle/working/
possible_paths = [
    '/kaggle/input/k9mistralready/k-9_Mistral_Ready.csv', # Exact path from your latest screenshot
    '/kaggle/working/k-9_Mistral_Ready.csv',
    '/kaggle/input/k-9_Mistral_Ready.csv'
]

CSV_PATH = None
for path in possible_paths:
    if os.path.exists(path):
        CSV_PATH = path
        break

# Custom search if the exact path isn't in the list above
if not CSV_PATH:
    for root, dirs, files in os.walk('/kaggle/input'):
        for file in files:
            if file == 'k-9_Mistral_Ready.csv':
                CSV_PATH = os.path.join(root, file)
                break

if CSV_PATH:
    print(f"✅ Found dataset at: {CSV_PATH}")
    issues_df = pd.read_csv(CSV_PATH)
    print(f"Loaded {len(issues_df)} issues ready for generation.")
else:
    print("❌ CSV NOT FOUND! Please ensure 'k-9_Mistral_Ready.csv' is uploaded to Kaggle.")
    print("Current contents of /kaggle/input:")
    for root, dirs, files in os.walk('/kaggle/input'):
        print(f"  {root}: {files}")
    raise FileNotFoundError("Please upload the CSV before continuing.")

✅ Found dataset at: /kaggle/input/datasets/oqtayhasanov/k-9dataset/k-9_Mistral_Ready.csv
Loaded 2935 issues ready for generation.


In [2]:
!pip install -q transformers bitsandbytes accelerate pandas tqdm

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left" # Required for batched generation

print("Loading model in 4-bit quantization...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

print(f"Model loaded! GPU memory used: {torch.cuda.memory_allocated()/1e9:.1f} GB")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 30.8 MB/s eta 0:00:00:00:0100:01


Loading tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

... (4 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded! GPU memory used: 1.6 GB


In [3]:
import random

PERSONA_TEMPLATES = [
    {"persona": "angry_short", "instruction": "Write a very short (1-2 sentences), angry 1-star Google Play Store review from a frustrated user who experienced this bug. Use informal language, maybe some caps. No technical jargon."},
    {"persona": "confused_user", "instruction": "Write a 2-3 sentence Google Play Store review from a confused, non-technical user who doesn't understand what went wrong. They just know something isn't working. Use simple vocabulary."},
    {"persona": "calm_reporter", "instruction": "Write a 2-3 sentence Google Play Store review from a calm user who matter-of-factly describes the problem. No anger, just stating what happened. Moderate detail."},
    {"persona": "sarcastic", "instruction": "Write a short, sarcastic Google Play Store review (1-2 sentences) from a user mocking the app for this issue. Dry humor."},
    {"persona": "desperate_user", "instruction": "Write a 2-3 sentence Google Play Store review from a desperate user who really needs the app to work but keeps hitting this bug. They are pleading for a fix."},
    {"persona": "technical_user", "instruction": "Write a 2-3 sentence Google Play Store review from a slightly technical user who describes the problem with some specificity but still uses everyday language, not developer jargon."},
    {"persona": "one_liner", "instruction": "Write a single short sentence Google Play Store review (under 15 words) that captures the essence of this bug from an end user perspective."},
    {"persona": "comparison_user", "instruction": "Write a 2-3 sentence Google Play Store review where the user compares this app unfavorably to a competitor because of this bug. Mention switching to another app."},
    {"persona": "broken_english", "instruction": "Write a 1-2 sentence Google Play Store review from a non-native English speaker with slightly broken grammar describing this bug. Keep it natural and respectful."},
    {"persona": "detailed_complaint", "instruction": "Write a 3-4 sentence detailed Google Play Store review describing exactly what happens when the user encounters this bug, what they expected instead, and that they want it fixed in the next update."}
]

def build_prompt(issue_title, issue_body, persona_template):
    system_msg = (
        "You are simulating real Google Play Store user reviews for an email app called K-9 Mail (Thunderbird for Android). "
        "Write ONLY the review text. No labels, no ratings, no explanations, no quotes around it. "
        "The review must sound like a real user wrote it, not a developer and not an AI. "
        "Keep it natural and authentic. Do NOT mention bug numbers, issue IDs, or GitHub."
    )
    
    # Truncate body if it's too long to save context window
    body_excerpt = str(issue_body)[:800] 
    issue_context = f"Title: {issue_title}\nDescription: {body_excerpt}"

    user_msg = (
        f"Here is a real bug report from the app's issue tracker:\n\n"
        f"{issue_context}\n\n"
        f"{persona_template['instruction']}\n\n"
        f"Write ONLY the review text, nothing else:"
    )

    return f"[INST] {system_msg}\n\n{user_msg} [/INST]"

def generate_review_batch(prompts, max_new_tokens=150, temperature=0.9, top_p=0.95):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    input_length = inputs.input_ids.shape[1]
    generated_tokens = outputs[:, input_length:]
    
    reviews = []
    for gen in generated_tokens:
        review = tokenizer.decode(gen, skip_special_tokens=True).strip()
        review = review.strip('"').strip("'").strip()
        
        for prefix in ["Review:", "review:", "Rating:", "Here is", "Here's"]:
            if review.lower().startswith(prefix.lower()):
                review = review[len(prefix):].strip()
                review = review.lstrip(':').strip()
        reviews.append(review)

    return reviews

In [4]:
import json, gc, os
from tqdm import tqdm
import pandas as pd
import torch
import random

# Updated file paths for K-9 Mail
CHECKPOINT_PATH = "/kaggle/working/checkpoint_k9_reviews.json"
CSV_OUT_PATH = "/kaggle/working/synthetic_reviews_k9.csv"
BATCH_SIZE = 10 # Process 10 issues at a time

all_results = []
completed_issue_ids = set()

# Check for existing checkpoint to resume progress
if os.path.exists(CHECKPOINT_PATH):
    print("Found existing checkpoint!")
    with open(CHECKPOINT_PATH, 'r') as f:
        checkpoint = json.load(f)
    all_results = checkpoint.get('results', [])
    completed_issue_ids = set(checkpoint.get('completed_ids', []))
    print(f"RESUMED: {len(completed_issue_ids)} issues processed.")
else:
    print("Starting fresh.")

# Filter out issues that have already been processed
remaining = issues_df[~issues_df['id'].isin(completed_issue_ids)].copy()
print(f"Remaining issues to process: {len(remaining)}")

# Convert remaining to list of dicts for easier batch slicing
remaining_records = remaining.to_dict('records')

for i in tqdm(range(0, len(remaining_records), BATCH_SIZE), desc="Generating Batches"):
    batch_records = remaining_records[i:i + BATCH_SIZE]
    
    batch_prompts = []
    batch_metadata = []
    
    # Setup the batch: 1 random persona per issue
    for row in batch_records:
        persona = random.choice(PERSONA_TEMPLATES)
        # build_prompt should now be the K-9 Mail version updated in Cell 3
        prompt = build_prompt(row['title'], row['clean_body'], persona)
        
        batch_prompts.append(prompt)
        batch_metadata.append({
            'issue_id': int(row['id']),
            'issue_number': int(row['number']),
            'issue_title': row['title'],
            'issue_state': row['state'],
            'persona': persona['persona']
        })

    try:
        # Generate all reviews in the batch simultaneously using the GPU
        generated_reviews = generate_review_batch(batch_prompts)
        
        # Map generated text back to its respective issue metadata
        for meta, review in zip(batch_metadata, generated_reviews):
            if len(review) >= 5: # Ensure the review isn't empty or a glitch
                meta['synthetic_review'] = review
                all_results.append(meta)
                completed_issue_ids.add(meta['issue_id'])
                
    except Exception as e:
        print(f"\nError processing batch starting at index {i}: {e}")
        continue

    # Periodic checkpointing every 5 batches (50 issues)
    if (i // BATCH_SIZE) % 5 == 0:
        with open(CHECKPOINT_PATH, 'w') as f:
            json.dump({'results': all_results, 'completed_ids': list(completed_issue_ids)}, f)
        pd.DataFrame(all_results).to_csv(CSV_OUT_PATH, index=False)
        
        # Memory management to prevent OOM errors on Kaggle
        gc.collect()
        torch.cuda.empty_cache()

# Final save upon completion
with open(CHECKPOINT_PATH, 'w') as f:
    json.dump({'results': all_results, 'completed_ids': list(completed_issue_ids)}, f)
pd.DataFrame(all_results).to_csv(CSV_OUT_PATH, index=False)

print(f"\nDONE! {len(all_results)} K-9 Mail reviews generated.")
print(f"Final dataset saved to: {CSV_OUT_PATH}")

Starting fresh.
Remaining issues to process: 2935



Generating Batches: 100%|██████████| 294/294 [2:30:17<00:00, 30.67s/it]


DONE! 2935 K-9 Mail reviews generated.
Final dataset saved to: /kaggle/working/synthetic_reviews_k9.csv


In [1]:
import pandas as pd

for f, name in [
    ("synthetic_reviews_brave__1_.csv", "brave"),
    ("synthetic_reviews_signal.csv", "signal"),
    ("synthetic_reviews_anki.csv", "anki"),
    ("synthetic_reviews_k9.csv", "k9"),
]:
    df = pd.read_csv(f)
    sample = df.sample(100, random_state=42)
    sample.to_csv(f"sample_{name}_100.csv", index=False)
    print(f"{name}: sampled 100")

FileNotFoundError: [Errno 2] No such file or directory: 'synthetic_reviews_brave__1_.csv'